<a href="https://colab.research.google.com/github/Hirunidiv/computer-vision-practice/blob/main/Model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

unzip data file

In [ ]:
import os

# 1. Look for any zip files in the current folder
files = [f for f in os.listdir('.') if f.endswith('.zip')]

if not files:
    print("❌ No zip files found! Look at the left sidebar (📁). Is your file inside a folder named 'sample_data'?")
else:
    # 2. Select the first zip file found and unzip it
    zip_file = files[0]
    print(f"📦 Found file: {zip_file}. Unzipping now...")
    os.system(f"unzip {zip_file} -d /content/my_data")
    print("✅ Done! Refresh your file tab.")


📦 Found file: model_data.zip. Unzipping now...
✅ Done! Refresh your file tab.


Define the Dataset Class

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class HairDamageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = ['weak', 'moderate', 'high', 'severe']
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        self.image_paths = []
        self.labels = []

        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.exists(cls_dir):
                print(f"Warning: Folder not found - {cls_dir}")
                continue
            for img_name in os.listdir(cls_dir):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(cls_dir, img_name))
                    self.labels.append(self.class_to_idx[cls])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        image = np.array(image)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        label = self.labels[idx]
        return image, label

Define Transforms + Create Datasets

In [ ]:
# Corrected Transforms
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.GaussNoise(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.15, rotate_limit=30, p=0.7),
    A.CoarseDropout(max_holes=12, max_height=25, max_width=25, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_476/110643738.py:10: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=12, max_height=25, max_width=25, p=0.5),


In [ ]:
import os
from torch.utils.data import DataLoader, WeightedRandomSampler

# Define data paths
base_data_path = '/content/my_data'
train_path = '/content/my_data/model_data/train'
val_path = '/content/my_data/model_data/val'
test_path = '/content/my_data/model_data/test'

# Instantiate Datasets
train_dataset = HairDamageDataset(train_path, transform=train_transform)
val_dataset   = HairDamageDataset(val_path,   transform=val_transform)
test_dataset  = HairDamageDataset(test_path,  transform=val_transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Weighted Sampler for imbalance
labels = train_dataset.labels
class_counts = np.bincount(labels)
class_weights = 1. / class_counts
sample_weights = [class_weights[label] for label in labels]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# Weighted sampler should be defined before DataLoader, as it uses train_dataset
# Calculate class weights for imbalance (assuming this block is run after the previous setup cell for sampler)
# If running this cell independently, ensure sampler is defined or remove it from DataLoader here

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("✅ Datasets and loaders updated with normalization!")

Train samples: 151
Val samples:   33
Test samples:  32
✅ Datasets and loaders updated with normalization!


In [ ]:
import pandas as pd
print("Train distribution:")
print(pd.Series(train_dataset.labels).value_counts().sort_index())

Train distribution:
0    10
1    84
2    44
3    13
Name: count, dtype: int64


Create DataLoaders + Handle Imbalance (Weighted Sampling)

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

# Calculate class weights for imbalance
labels = train_dataset.labels
class_counts = np.bincount(labels)
class_weights = 1. / class_counts
sample_weights = [class_weights[label] for label in labels]

# Weighted sampler (helps a lot with imbalance)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders
batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("✅ DataLoaders created successfully!")
print(f"Training batches per epoch: {len(train_loader)}")

✅ DataLoaders created successfully!
Training batches per epoch: 10


Define the Model

In [ ]:
import timm
import torch.nn as nn

# Best simple & strong model for your case
model = timm.create_model('convnext_tiny', pretrained=True, num_classes=4)

# OR use ResNet50 if you prefer
# model = timm.create_model('resnet50', pretrained=True, num_classes=4)

model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Model loaded!")
print(f"Model name: {model.__class__.__name__}")

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

✅ Model loaded!
Model name: ConvNeXt


Training Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# criterion = nn.CrossEntropyLoss()
# Weighted Loss
class_weights = torch.tensor([1/10, 1/84, 1/44, 1/13], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Fixed scheduler (removed verbose)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max',
                                                 factor=0.5, patience=5)

best_f1 = 0.0
patience = 15
counter = 0
num_epochs = 50

save_path = "/content/drive/MyDrive/HairDamageProject/best_model.pth"

print("✅ Training setup ready!")

Using device: cuda
✅ Training setup ready!


In [ ]:
import os

save_dir = "/content/drive/MyDrive/HairDamageProject"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "best_model.pth")
print(f"✅ Save directory ready: {save_path}")

✅ Save directory ready: /content/drive/MyDrive/HairDamageProject/best_model.pth


Main Training Loop

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_preds, train_true = [], []

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_preds.extend(preds.cpu().numpy())
        train_true.extend(labels.cpu().numpy())

    # Validation
    model.eval()
    val_loss = 0.0
    val_preds, val_true = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    # Calculate metrics
    train_acc = accuracy_score(train_true, train_preds)
    val_acc = accuracy_score(val_true, val_preds)
    val_f1 = f1_score(val_true, val_preds, average='macro')

    print(f"\nEpoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")
    print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")

    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)
        print(f"✅ Best model saved! (F1: {best_f1:.4f})")
        counter = 0
    else:
        counter += 1

    scheduler.step(val_f1)

    if counter >= patience:
        print("Early stopping triggered!")
        break

Epoch 1/50: 100%|██████████| 10/10 [00:13<00:00,  1.37s/it]



Epoch 1 | Train Loss: 1.5655 | Val Loss: 1.6054
Train Acc: 0.2318 | Val Acc: 0.0909 | Val Macro F1: 0.0417
✅ Best model saved! (F1: 0.0417)


Epoch 2/50: 100%|██████████| 10/10 [00:04<00:00,  2.14it/s]



Epoch 2 | Train Loss: 1.2432 | Val Loss: 1.4912
Train Acc: 0.2185 | Val Acc: 0.0606 | Val Macro F1: 0.0294


Epoch 3/50: 100%|██████████| 10/10 [00:05<00:00,  1.74it/s]


...


Epoch 14 | Train Loss: 1.2340 | Val Loss: 1.5301
Train Acc: 0.1987 | Val Acc: 0.0606 | Val Macro F1: 0.0286


Epoch 15/50: 100%|██████████| 10/10 [00:04<00:00,  2.11it/s]



Epoch 15 | Train Loss: 1.1539 | Val Loss: 1.4663
Train Acc: 0.2318 | Val Acc: 0.0909 | Val Macro F1: 0.0417


Epoch 16/50: 100%|██████████| 10/10 [00:05<00:00,  1.71it/s]



Epoch 16 | Train Loss: 1.1571 | Val Loss: 1.4997
Train Acc: 0.2715 | Val Acc: 0.0606 | Val Macro F1: 0.0286
Early stopping triggered!
